In [1]:
#%pip install pandas lxml html5lib --quiet
import pandas as pd
url = "https://www.bbc.co.uk/sport/football/world-cup/table"
try:
    tables = pd.read_html(url)
    
    for index, group_df in enumerate(tables[:-1]):
        #print(f"\n--- Group {chr(65 + index)} Live Table ---")
        
        group_df.columns = [str(col).strip() for col in group_df.columns]
        
        group_df['Team'] = group_df['Team'].str.replace(r'^\d+\.?\s*', '', regex=True)

        #print(group_df[['Team', 'Played', 'Points']])

except Exception as e:
    print("Error fetching live data")

In [2]:
family = {'Eagle 1' : ['Brazil', 'Turkey', 'Ivory Coast', 'South Korea', 'Paraguay', 'United States', 'Austria', 'Portugal'], 
          'Eagle 2' : ['Belgium', 'Australia', 'France', 'Algeria', 'Morocco', 'Canada', 'Czech Republic', 'Iraq'], 
          'Eagle 3' : ['Panama', 'Ecuador', 'Argentina', 'Japan', 'England', 'Ghana', 'Iran', 'South Africa'], 
          'Eagle 4' : ['Qatar', 'Germany', 'Senegal', 'Colombia', 'Curaçao', 'Switzerland', 'Congo DR', 'Netherlands'], 
          'Eagle 5' : ['Scotland', 'Saudi Arabia', 'Tunisia', 'Bosnia-Herzegovina', 'Haiti', 'Mexico', 'Croatia', 'Egypt'], 
          'Eagle 6' : ['Jordan', 'Cape Verde', 'Norway', 'Spain', 'Uruguay', 'Sweden', 'Uzbekistan', 'New Zealand'],}

In [3]:
all_teams = []
for group_df in tables[:-1]:
    group_df.columns = [str(col).strip() for col in group_df.columns]
    group_df['Team'] = group_df['Team'].str.replace(r'^\d+\.?\s*', '', regex=True)
    all_teams.append(group_df[['Team', 'Played', 'Goal Difference', 'Points']])

all_teams_df = pd.concat(all_teams, ignore_index=True)

team_to_member = {team: member for member, teams in family.items() for team in teams}

all_teams_df['Member'] = all_teams_df['Team'].map(team_to_member)

family_table = all_teams_df.groupby('Member')[['Played', 'Goal Difference', 'Points']].sum()

family_table['Points per match'] = round(family_table['Points'] / family_table['Played'], 2)

family_table = family_table.sort_values('Points', ascending =False)

print(family_table)

         Played  Goal Difference  Points  Points per match
Member                                                    
Eagle 1      22               12      36              1.64
Eagle 3      19                9      29              1.53
Eagle 4      21               -1      29              1.38
Eagle 2      20                1      27              1.35
Eagle 5      21              -17      24              1.14
Eagle 6      17               -4      19              1.12


The following chunk was in part generated using Claude to fine tune the HTML

In [8]:
from IPython.display import HTML, Javascript, display
import json

rows = []
for member, row in family_table.iterrows():
    rows.append({
        "Member": member,
        "Played": int(row["Played"]),
        "GD":     int(row["Goal Difference"]),
        "Points": int(row["Points"]),
        "PPM":    float(row["Points per match"])
    })

data_json = json.dumps(rows)

html = """
<style>
table{width:100%;border-collapse:collapse;font-size:14px;font-family:sans-serif}
th,td{padding:10px 14px;text-align:left;border-bottom:1px solid #e5e5e5}
td:not(:first-child),th:not(:first-child){text-align:right}
thead th{font-weight:500;font-size:13px;color:#888;white-space:nowrap}
tbody tr:hover td{background:#f9f9f9}
.sort-btn{display:inline-flex;align-items:center;gap:6px;padding:6px 14px;
          border:1px solid #ddd;border-radius:6px;background:#fff;
          font-size:13px;cursor:pointer;margin-right:8px;margin-bottom:16px}
.sort-btn.active{background:#f0f0f0;border-color:#bbb;font-weight:500}
.rank{color:#aaa;font-size:12px}
.badge{background:#f3f3f3;border-radius:4px;padding:2px 7px;font-size:12px;color:#666}
.pos{color:green}.neg{color:#c0392b}
</style>

<div>
  <button class="sort-btn active" id="btn-pts">Sort by points</button>
  <button class="sort-btn" id="btn-ppm">Sort by pts/match</button>
</div>

<table>
  <thead>
    <tr>
      <th style="text-align:left">Pos</th>
      <th style="text-align:left">Member</th>
      <th>Played</th><th>GD</th><th>Points</th><th>Pts/match</th>
    </tr>
  </thead>
  <tbody id="tbody"></tbody>
</table>
"""

js = """
(function() {
  var rawData = """ + data_json + """;
  var currentSort = 'Points';
  var desc = true;

  function sortBy(key) {
    if (currentSort === key) desc = !desc;
    else { currentSort = key; desc = true; }
    render();
  }

  function render() {
    var sorted = rawData.slice().sort(function(a, b) {
      var va = currentSort === 'Points' ? a.Points : a.PPM;
      var vb = currentSort === 'Points' ? b.Points : b.PPM;
      return desc ? vb - va : va - vb;
    });

    document.getElementById('btn-pts').className = 'sort-btn' + (currentSort === 'Points' ? ' active' : '');
    document.getElementById('btn-ppm').className = 'sort-btn' + (currentSort === 'PPM' ? ' active' : '');

    document.getElementById('tbody').innerHTML = sorted.map(function(r, i) {
      var gdSign = r.GD >= 0 ? '+' : '';
      var gdClass = r.GD >= 0 ? 'pos' : 'neg';
      return '<tr>' +
        '<td><span class="rank">' + (i + 1) + '</span></td>' +
        '<td style="font-weight:500">' + r.Member + '</td>' +
        '<td><span class="badge">' + r.Played + '</span></td>' +
        '<td class="' + gdClass + '">' + gdSign + r.GD + '</td>' +
        '<td>' + r.Points + '</td>' +
        '<td>' + r.PPM.toFixed(2) + '</td>' +
        '</tr>';
    }).join('');
  }

  document.getElementById('btn-pts').addEventListener('click', function() { sortBy('Points'); });
  document.getElementById('btn-ppm').addEventListener('click', function() { sortBy('PPM'); });

  render();
})();
"""

display(HTML(html))
display(Javascript(js))

Pos,Member,Played,GD,Points,Pts/match


<IPython.core.display.Javascript object>